## Exploration données batiments

[COLLECTE] Données date construction bâtiments
https://mattermost.services.dataforgood.fr/boards/team/4g8gayrar3r5d8ofobpk9kaxkr/bjjws538jrir4zxuh35r9kr35qo/vpki6pz3czj89fkjgth3kbq659r/cfti645hma3nbjn6woxwp7t9q9o


In [ ]:
import pandas as pd
import polars as pl
import numpy as np
import geopandas as gpd
import pyogrio
import os
import folium # oOr map viz

# Source 1: Données Logement INSEE


In [ ]:
bat = pd.read_csv('../csv/DS_RP_LOGEMENT_PRINC_2022_data.csv', delimiter=';')
bat.info()

In [ ]:
bat_bdx = bat[
    (bat.GEO == '13055') &
    # (bat.GEO_OBJECT == 'COM') &
    (bat.TIME_PERIOD == 2022)]

In [ ]:
bat_bdx.GEO_OBJECT.unique()

In [ ]:
bat_bdx = bat[
    (bat.GEO == '33063') &
    (bat.GEO_OBJECT == 'COM') &
    (bat.TIME_PERIOD == 2022)]

bat_bdx.head()

In [ ]:
bat_bdx[~(bat_bdx.BUILD_END == '_T')]

In [ ]:
bat.head()

In [ ]:
bat_com = bat[
    ~(bat.BUILD_END == '_T') &      # On ignore les totaux qui donnent des découpages selon d'autres axes d'analyses
    ~(bat.TDW == '_T') &            # Type de logement (On ignore le total pour ne pas compter deux fois) 
    (bat.GEO_OBJECT == 'COM') &     # On ne garde que les communes (attention pour les PLM)
    (bat.TIME_PERIOD == 2022)     # Pour l'instant on ne garde que la denière année (2022)
] 
bat_com.info()

In [ ]:
del bat #save memory

In [ ]:
bat_com_agg = bat_com[['GEO', 'BUILD_END','OBS_VALUE']].groupby(['GEO', 'BUILD_END']).sum('OBS_VALUE').reset_index()
bat_com_agg = bat_com_agg.pivot_table(index='GEO', columns=['BUILD_END'], values='OBS_VALUE').reset_index()
year_col = [col for col in bat_com_agg.columns.values if col.startswith('Y')]
bat_com_agg['Total'] = bat_com_agg[year_col].sum(axis=1)
bat_com_agg.head()

In [ ]:
bat_com_agg.head()

In [ ]:
bat_com_agg[bat_com_agg.GEO == '33063']

# Source 2: Données Inondations (Géorisques)


Fichier par département pour les zones TRI Territoire à Risque d'Inondation de GeoRisques (2020): https://www.data.gouv.fr/datasets/territoire-a-risque-dinondation-tri-du-sig-directive-inondation-france-metropolitaine-rapportage-2020-241

Fichier au format Shapefile

Layers:

- national_flood_extents : Les polygones des zones inondables par niveau de risque
- national_water_heights : Les classes de hauteurs d'eau.
- national_assets : Les enjeux (bâtiments, infrastructures).
- national_tri_communes : Les périmètres des communes des TRI. (Une ligne par commune+TRI qui la concerne)

Pour lire un layer spécifique:
`gpd.read_file("path_to.gpkg", layer="layer_name")`

Type Inondations:

- 01: Débordements de cours d’eau
- 02: Ruissellement : scénarios moyen et extrême seulement
- 03: Submersion marine
- 04: Remontées de nappes (débordements des eaux souterraines) : scénario extrême seulement


In [ ]:
# TRI_FRANCE_FILE = "../csv_large/national_tri_2020.gpkg" # Original crazy detailed 15Gb file
TRI_FRANCE_FILE = "../csv_large/national_tri_2020_optimized.gpkg" # Only XGb
PYOGRIO_USE_ARROW=1 #Faster read/write

In [ ]:
# Liste les couches et leurs types de géométrie
layers = pyogrio.list_layers(TRI_FRANCE_FILE)
print(layers)
# Obtient les informations détaillées (colonnes, types) d'une couche précise
info = pyogrio.read_info(TRI_FRANCE_FILE, layer="national_tri_communes")
print(info['fields'])
info2 = pyogrio.read_info(TRI_FRANCE_FILE, layer="national_flood_extents")
print(info['fields'])

In [ ]:
# Charge les données
com = gpd.read_file("../csv_large/national_tri_2020.gpkg", layer="national_tri_communes")#, rows=100)
flood_head = gpd.read_file("../csv_large/national_tri_2020.gpkg", layer="national_flood_extents", rows=1000)
# flood = gpd.read_file("../csv_large/national_tri_2020.gpkg", layer="national_flood_extents")

In [ ]:
com.head()

In [ ]:
# com.explore()

In [ ]:
com.groupby('code_insee').count().sort_values('id', ascending=False)

In [ ]:
com[com.code_insee == '92012']

In [ ]:
com[com.id_tri == 'FRH_TRI_METROPOLEFRANCILIENNE'].drop_duplicates('geometry').plot()

In [ ]:
flood.head()

In [ ]:
flood.head(1000).explore(column='scenario', cmap='viridis')

# Source 3: Base bâtiments BNDB


Description des tables ici: https://bdnb.io/schema/latest/bdnb_v07/index.html

- **batiment_groupe**: Groupes de bâtiments au sens de la BDNB.

Un groupe de bâtiments est composé de 1 à n bâtiments fonciers et de 1 à m bâtiments constructions.
Ces entités sont créées pour minimiser n et m, afin de produire des groupes aussi précis que possible, tout en garantissant une adéquation entre : - le contenant : enceintes physiques des bâtiments (batiment_construction), - et le contenu : bâtiments fonciers qui documentent usages, surfaces et nombre de locaux.

Un groupe de bâtiments est toujours situé sur une même unité foncière (ensemble de parcelles adjacentes appartenant à un même propriétaire).

Cette entité, qui assure la cohérence entre contenant et contenu, constitue le socle de la BDNB.
La majorité des informations présentes dans la base sont définies à cette échelle : batiment_groupe.

- **batiment_groupe_risques** = Table regroupant différents indicateurs de risques (sismique, incendie, radon et argile) associés aux batiment_groupe de la BDNB. Les aléas sismiques, radon et argiles proviennent des zonages issus du site georisques. Les catégories de familles incendies proviennent d’une prédiction simplifiée de la réglementation risque incendie (sur le résidentiel uniquement). Enfin, les catégories de bâtiments associées au risque sismique correspondent à des travaux similaires au risque incendie (sur le résidentiel uniquement).

- **batiment_groupe_ffo_bat**: Données issues du traitement des fichiers fonciers pour ce bâtiment.

Caractéristiques : - Les quantitatifs (nombre de locaux, surfaces) sont calculés comme la somme des locaux et de leurs surfaces sur l’ensemble des bâtiments fonciers idbat constitutifs du groupe de bâtiments de la BDNB. - Les données d’usages sont une synthèse des différents usages identifiés dans les fichiers fonciers.

- **batiment_groupe_synthese_propriete_usage**: Table de synthèse des informations de propriété et d’usage des bâtiments.

Sources : Cette table consolide les données issues de multiples sources : - RPLS - Fichiers fonciers - Registre des copropriétés - ESPACE (projet de géolocalisation des établissements publics en partenariat avec le CEREMA) - Fichier locaux des personnes morales de la DGFiP - SIRENE

Objectif : Fournir les informations principales sur la propriété et l’usage des groupes de bâtiments.

NB : Cette table introduit le nouvel indicateur usage_principal_bdnb_open, qui remplace l’ancienne variable usage_niveau_1_txt issue des seuls fichiers fonciers.
Il est recommandé d’utiliser usage_principal_bdnb_open à la place de la donnée foncière brute.


## Approach 1: API (Doesnt work)

Using the API with Open (free) plan we cannot fetch large amount of data. But this is an example of how it works below


In [ ]:
import requests
r = requests.get(f'https://api.bdnb.io/v1/bdnb/donnees/batiment_groupe_risques',
                 params={   
                         'select': 'batiment_groupe_id, code_departement_insee, alea_argile, classe_famille_incendie_residentiel',
                         'batiment_groupe_id': 'eq.bdnb-bg-GWH9-79G1-B81G',
                        #  'code_departement_insee': '33',
                         'limit': 5}                 
                 )
r.json()

In [ ]:
import requests
r = requests.get('https://api.bdnb.io/v1/bdnb/geocodage?',
             params={'q': '57 rue Reinette Bordeaux'})
r.json()

In [ ]:
import requests
r = requests.get(f'https://api.bdnb.io/v1/bdnb/donnees/batiment_groupe_complet/adresse',
                     params={'limit': 5,  # limite le nombre de réponses à 5 résultats
                             'cle_interop_adr': 'eq.33063_7850_00057'})

r.json()

## Approach 2: Local CSV (works)

We can use the BDNB dumps by département (PoC) or France entière that can be found here: https://bdnb.io/download/

Note: Because the file is huge (35Gb compressed) we leverage polars for data manipulation vs pandas


In [ ]:
# Load necessary imports and variables
import zipfile
import tarfile
import os
from pathlib import Path
import glob
import shutil
import shapely
from shapely import wkt
from shapely.geometry import Point


# Configuration

# PoC (Ain)
# ARCHIVE_PATH = "../csv/open_data_millesime_2025-07-a_dep01_csv.zip"
# EXTRACT_DIR = "../csv/extracted_dep01"
# OUTPUT_FILE = "../csv/dep01_houses_poc.csv"

# France entière
ARCHIVE_PATH = Path("../csv_large/open_data_millesime_2025-07-a_france_csv.tar.gz")
TRI_GPKG = Path("../csv_large/national_tri_2020_optimized.gpkg")
SOURCE_DIR = Path("../csv_large/extracted_france/csv")
PARTITIONS_DIR = Path("../csv_large/bdnb_partitions")
OUTPUT_FILE_CLEAN = Path("../csv_large/france_houses_clean.parquet")
OUTPUT_FILE_AGG = Path("../csv/france_houses_agg.parquet")
OUTPUT_DIR = Path("../csv_large/bdnb_processed")
TEMP_OUTPUT_DIR = Path("../csv_large/bdnb_processed_chunks")



# Relevant files in the archive (standard paths for France dump often have ./ prefixes or different nesting)
# We will search for these suffixes
REQUIRED_SUFFIXES = [
    "batiment_groupe.csv",
    "batiment_groupe_risques.csv",
    "batiment_groupe_ffo_bat.csv",
    "batiment_groupe_synthese_propriete_usage.csv"
]


In [ ]:
# Extracting tables from main Zip file
def get_file_map(archive_list):
    """Maps suffix to the full path in the archive."""
    file_map = {}
    for actual_path in archive_list:
        for suffix in REQUIRED_SUFFIXES:
            if actual_path.endswith(suffix):
                file_map[suffix] = actual_path
    return file_map

def extract_files():
    print(f"Opening archive {os.path.basename(ARCHIVE_PATH)}...")
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    
    if ARCHIVE_PATH.endswith('.zip'):
        with zipfile.ZipFile(ARCHIVE_PATH, 'r') as z:
            file_map = get_file_map(z.namelist())
            for name, actual_path in file_map.items():
                print(f"  Extracting {actual_path}...")
                z.extract(actual_path, EXTRACT_DIR)
    
    elif ARCHIVE_PATH.endswith('.tar.gz') or ARCHIVE_PATH.endswith('.tgz'):
        with tarfile.open(ARCHIVE_PATH, 'r:gz') as t:
            file_map = get_file_map(t.getnames())
            for name, actual_path in file_map.items():
                print(f"  Extracting {actual_path}...")
                t.extract(actual_path, EXTRACT_DIR)
    
    return file_map

If we fetch and work with the geometries, it will be to slow to process all the data in one go. The new approach below is to process department by department.
4 steps:

1. Fetch and join the BDNB tables
2. Compute centroids for all bâtiments
3. Spatial join with the TRI polygons (Territoires à Risque d'Inondations)
4. Agg by communes + risques RGA / Inondations


In [ ]:
# 1. Partition the data by departement and join tables

# List of files we need
FILES = {
    'usage': 'batiment_groupe_synthese_propriete_usage.csv',
    'base': 'batiment_groupe.csv',
    'risques': 'batiment_groupe_risques.csv',
    'ffo': 'batiment_groupe_ffo_bat.csv'
}
paths = {key: SOURCE_DIR / filename for key, filename in FILES.items()}

def get_construction_period():
    """Returns a Polars expression for construction period grouping (INSEE Standard)."""
    return (
        pl.when(pl.col("annee_construction").is_null()).then(pl.lit("Inconnue"))
        .when(pl.col("annee_construction") < 1945).then(pl.lit("Avant 1945"))
        .when(pl.col("annee_construction") < 1976).then(pl.lit("1945-1975"))
        .when(pl.col("annee_construction") < 2020).then(pl.lit("1976-2020"))
        .otherwise(pl.lit("Après 2020"))
        .alias("periode_construction")
    )

def partition_data():
    print(f"\n--- Phase 1: Partitioning BDNB Data by Department ---")
    print(f"Reading from {SOURCE_DIR}")
    
    # 1. Define the list of departments for Metropolitan France
    # 01-19, 2A, 2B, 21-95
    print("Setting target scope: Metropolitan France...")
    deps_numeric = [str(i).zfill(2) for i in range(1, 96) if i != 20]
    deps = sorted(deps_numeric + ["2A", "2B"])
    print(f"Targeting {len(deps)} departments (01-95, including 2A/2B).")

    # 2. Iterate through departments
    with pl.StringCache():
        for dep in deps:
            out_path = PARTITIONS_DIR / f"code_departement_insee={dep}"
            target_file = out_path / "data.parquet"
            
            if target_file.exists():
                print(f"  -> Skipping Department {dep} (Already partitioned)")
                continue
                
            print(f"  -> Processing Department {dep}...")
            
            # Create isolated scans FOR THIS DEPARTMENT ONLY
            # By applying the filter immediately on the scan, Polars can push down the predicate
            # to the CSV reader, massively reducing the size of the data before the join.
            
            q_base = (
                pl.scan_csv(paths['base'], separator=';', 
                           infer_schema_length=10000, 
                           schema_overrides={"code_departement_insee": pl.String, "code_commune_insee": pl.String})
                .with_columns(pl.col("code_departement_insee").cast(pl.String).str.zfill(2))
                .filter(pl.col("code_departement_insee") == dep)
                .select(["batiment_groupe_id", "code_departement_insee", "code_commune_insee", "geom_groupe"])
            )
            
            # For the other files, it might be faster to just load the whole thing into memory once 
            # if they are small enough, but joining the scans is safer memory-wise.
            q_usage = (
                pl.scan_csv(paths['usage'], separator=';', infer_schema_length=10000)
                .filter(pl.col("usage_principal_bdnb_open") == "Résidentiel individuel")
                .select("batiment_groupe_id")
            )
            q_risks = (
                pl.scan_csv(paths['risques'], separator=';', infer_schema_length=10000)
                .select(["batiment_groupe_id", "alea_argile"])
            )
            q_ffo = (
                pl.scan_csv(paths['ffo'], separator=';', infer_schema_length=10000)
                .select(["batiment_groupe_id", "annee_construction"])
            )
            
            # The join focuses ONLY on the department's geometry rows
            plan = (
                q_base
                .join(q_usage, on="batiment_groupe_id", how="inner") # Inner join immediately filters to residentiel
                .join(q_risks, on="batiment_groupe_id", how="left")
                .join(q_ffo, on="batiment_groupe_id", how="left")
                .with_columns([
                    pl.col("code_commune_insee").cast(pl.String).str.zfill(5),
                    get_construction_period(),
                    pl.col("alea_argile").fill_null("Nul").cast(pl.Categorical),
                ])
                .drop("annee_construction")
            )
            
            try:
                # This should now execute relatively quickly per department since the 
                # massive geometries are only loaded for the specific department
                out_path.mkdir(exist_ok=True, parents=True)
                plan.sink_parquet(target_file)
            except Exception as e:
                print(f"     Error processing {dep}: {e}")



In [ ]:
## Runs the data join + partitioning

PARTITIONS_DIR.mkdir(parents=True, exist_ok=True)

partition_data()

In [ ]:
# 2. Compute Centroids and drop the heavy batiment polygons

def compute_centroids_for_partition(dep_dir: str):
    """Loads a BDNB partition, calculates centroids, drops WKT, and overwrites the parquet."""
    
    dep_name = os.path.basename(dep_dir)
    dep_code = dep_name.split('=')[-1]
    
    # We look for the data.parquet file inside the partition folder
    data_files = glob.glob(os.path.join(dep_dir, "*.parquet"))
    
    if not data_files:
        print(f"Skipping {dep_code} - no data files found.")
        return
        
    print(f"\n--- Computing Centroids for Department {dep_code} ---")
    
    for f in data_files:
        try:
            # 1. Load Partition
            df = pl.read_parquet(f)
            
            if "geom_groupe" not in df.columns:
                print(f"  -> {os.path.basename(f)}: Already processed (no 'geom_groupe'). Skipping.")
                continue
                
            print(f"  -> {os.path.basename(f)}: Processing {len(df)} rows...")
            
            # 2. Extract WKT and compute Centroids
            # We use Shapely to parse and compute the centroid
            # Then we convert to WKB (binary) for efficient storage in Parquet
            wkt_list = df["geom_groupe"].to_list()
            
            def get_wkb_centroid(wkt_str):
                if not wkt_str: return None
                try:
                    geom = wkt.loads(wkt_str)
                    return geom.centroid.wkb
                except:
                    return None
            
            # Vectorized-ish approach in Python
            centroids_wkb = [get_wkb_centroid(g) for g in wkt_list]
            
            # 3. Update DataFrame
            df_new = df.with_columns(
                pl.Series("centroid_wkb", centroids_wkb, dtype=pl.Binary)
            ).drop("geom_groupe")
            
            # 4. Save Back
            df_new.write_parquet(f)
            print(f"     Done. Replaced 'geom_groupe' with 'centroid_wkb'.")
            
        except Exception as e:
            print(f"     Error processing {f}: {e}")


In [ ]:
## Runs the centroids compute

# Find all partitioned folders
partitions = sorted(glob.glob(str(PARTITIONS_DIR / "code_departement_insee=*")))

if not partitions:
    print(f"No partitions found in {PARTITIONS_DIR}. Did Phase 1 succeed?")
else:
    for p in partitions:
        compute_centroids_for_partition(p)
print("\n--- Centroid Computation Complete for all partitions ---")


In [ ]:
# 3. Spatial sjoin to identify houses in TRI

def process_department(dep_dir: str):
    """Loads a BDNB partition, and spatial joins with TRI flood zones."""
    
    # Extract department code from directory name, e.g., 'code_departement_insee=01'
    dep_name = os.path.basename(dep_dir)
    dep_code = dep_name.split('=')[-1]
    
    out_file = TEMP_OUTPUT_DIR / f"processed_{dep_code}.parquet"
    if out_file.exists():
        print(f"Skipping {dep_code} - chunk already exists.")
        return
        
    print(f"\n--- Processing Department {dep_code} ---")
    
    ## 1. Load the BDNB Partition
    try:
        df_houses = pl.read_parquet(os.path.join(dep_dir, "*.parquet"))
    except Exception as e:
        print(f"Error loading BDNB partition for {dep_code}: {e}")
        return

    if df_houses.is_empty():
        print(f"No residential houses found for {dep_code}. Skipping.")
        return
        
    print(f"Loaded {len(df_houses)} houses.")
    
    ## 2. Load Centroids from WKB
    print("Loading pre-computed Centroids from WKB...")
    if "centroid_wkb" not in df_houses.columns:
        print(f"Error: 'centroid_wkb' not found in department {dep_code}. Did you run scripts/compute_centroids.py?")
        return
        
    # Speed: shapely.from_wkb is extremely fast on binary series
    geometries = [shapely.from_wkb(g) if g else None for g in df_houses["centroid_wkb"].to_list()]
    
    # Create GeoDataFrame
    gdf_houses = gpd.GeoDataFrame(
        df_houses.drop("centroid_wkb").to_pandas(), 
        geometry=geometries, 
        crs="EPSG:2154" # BDNB coordinates are in Lambert 93
    )
    
    ## 3. Load TRI Flood Zone Geometries
    print(f"Loading TRI Flood Zones for department {dep_code}...")
    try:
        where_clause = f"dep_code = '{dep_code}'"
        gdf_tri = gpd.read_file(
            TRI_GPKG, 
            layer="national_flood_extents",
            where=where_clause,
            engine="pyogrio",
            columns=["scenario_val", "typ_inond"]
        )
        print(f"Loaded {len(gdf_tri)} flood zones.")
    except Exception as e:
        print(f"Error loading TRI for {dep_code}: {e}")
        gdf_tri = gpd.GeoDataFrame()

    ## 4. Initialize columns and Spatial Join
    gdf_houses['scenario_inondation'] = "Aucun"
    gdf_houses['type_inondation'] = "Aucun"

    if gdf_tri.empty:
        print(f"No flood zones recorded for department {dep_code}.")
    else:
        print(f"Executing Spatial Join (sjoin) on {len(gdf_houses)} houses...")
        
        # Ensure CRS match
        if gdf_houses.crs != gdf_tri.crs:
            gdf_houses = gdf_houses.to_crs(gdf_tri.crs)
            
        # left join avoids dropping houses outside the flood zones
        joined = gpd.sjoin(gdf_houses, gdf_tri, how="left", predicate="intersects")
        
        # Priority Logic: Keep the highest risk scenario
        scenario_priority = {'01For': 0, '02Moy': 1, '04Fai': 2, 'Unknown': 3}
        joined['priority'] = joined['scenario_val'].map(scenario_priority).fillna(99)
        joined = joined.sort_values(by=['priority'])
        joined = joined[~joined.index.duplicated(keep='first')]
        
        # Populate the fields
        gdf_houses['scenario_inondation'] = joined['scenario_val'].fillna("Aucun")
        gdf_houses['type_inondation'] = joined['typ_inond'].fillna("Aucun")
        print("Join complete.")

    ## 5. Save Chunk
    print(f"Saving chunk for {dep_code}...")
    df_final = pl.from_pandas(gdf_houses.drop(columns=['geometry']))
    df_final.write_parquet(out_file)
    print("Done.\n")


def consolidate_results():
    print(f"\n--- Consolidating chunks into {OUTPUT_FILE_CLEAN} ---")
    chunks = glob.glob(str(TEMP_OUTPUT_DIR / "*.parquet"))
    if not chunks:
        print("No chunks found to consolidate.")
        return
        
    print(f"Merging {len(chunks)} fragments...")
    try:
        # Use scanning to merge potentially large fragmented results efficiently
        df_all = pl.scan_parquet(str(TEMP_OUTPUT_DIR / "*.parquet")).collect(streaming=True)
        df_all.write_parquet(OUTPUT_FILE_CLEAN)
        print(f"Successfully consolidated {len(df_all)} houses into {OUTPUT_FILE_CLEAN}")
        
        # Optional: Print summary
        print(df_all["scenario_inondation"].value_counts())
        
    except Exception as e:
        print(f"Consolidation failed: {e}")
                

In [ ]:
## Run Inondations lookup

TEMP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not TRI_GPKG.exists():
    print(f"CRITICAL ERROR: {TRI_GPKG} not found!")
else:
    # Find all partitioned folders
    partitions = glob.glob(str(PARTITIONS_DIR / "code_departement_insee=*"))

    if not partitions:
        print(f"No partitions found in {PARTITIONS_DIR}. Did Phase 1 succeed?")
    else:
        for p in partitions:
            process_department(p)

        # Final Merge
        consolidate_results()   

In [ ]:
# 4. Aggregations by commune
def aggregate_national():
    print(f"\n--- Phase 3: Final Aggregation ---")
    
    # 1. Scan the single consolidated master file
    if not OUTPUT_FILE_CLEAN.exists():
        print(f"Error: Master file {OUTPUT_FILE_CLEAN} not found. Ensure Phase 2 completed.")
        return
        
    print(f"Scanning consolidated master file: {OUTPUT_FILE_CLEAN}...")
    
    try:
        df_all = pl.scan_parquet(OUTPUT_FILE_CLEAN)
    except Exception as e:
        print(f"Error loading master file: {e}")
        return
    
    # 2. Aggregation Plan
    # We group by the commune, the construction period, the RGA risk, and the flood risk status
    # This distills millions of rows down to just combinations of counts per commune.
    print("Building aggregation plan...")
    
    agg_plan = (
        df_all
        .group_by(["code_commune_insee", "periode_construction", "alea_argile", "scenario_inondation", "type_inondation"])
        .agg(pl.len().alias("nb_maisons"))
        .sort(["code_commune_insee", "periode_construction"])
    )
    
    # 3. Execute
    print("Executing streaming aggregation (This may take a minute or two)...")
    df_final = agg_plan.collect(engine='streaming')
    
    # 4. Save and Preview
    print(f"\nNational Summary Preview:")
    print(df_final.head(10))
    print(f"\nTotal aggregated rows: {len(df_final)}")
    
    df_final.write_parquet(OUTPUT_FILE_AGG)
    print(f"\nFinal results saved to {OUTPUT_FILE_AGG}")

In [ ]:
## Run the aggregation
aggregate_national()

In [ ]:
# Manual check with a specific house I know is subject to flood risk
df = pd.read_parquet(OUTPUT_FILE_CLEAN)
df[df.batiment_groupe_id == 'bdnb-bg-TKBA-VHR2-P6P9']

In [ ]:
# Check with specific commune
df_agg = pd.read_parquet(OUTPUT_FILE_AGG)
df_agg[df_agg.code_commune_insee == '29006'].sort_values('nb_maisons', ascending=False).head(20)

Ce que l'on veut: une seule ligne par commune avec

- code insee
- total_maisons
- rga*[periode]*[niveau] (=4x4=16 colonnes)
- tri*[type]*[niveaux] (=4x4=16 colonnes)

Total 34 colonnes


In [ ]:
import pandas as pd
def flatten_commune_data(parquet_path):
    df = pd.read_parquet(parquet_path)
    
    # 1. Standardize/Clean values for column names
    # Mapping for periods
    p_map = {
        'Avant 1945': 'pre1945',
        '1945-1975': '1945_1975', 
        '1976-2020': '1976_2020',
        'Après 2020': 'post2020',
        'Inconnue': 'unk'
    }
    # Mapping for RGA and TRI Scenarios
    level_map = {
        'Nul': 'nul', 'Faible': 'faible', 'Moyen': 'moyen', 'Fort': 'fort',
        '01For': 'fort', '02Moy': 'moyen', '04Fai': 'faible', 'Aucun': 'nul'
    }
    # Mapping for TRI Types
    type_map = {'01': 't01', '02': 't02', '03': 't03', 'Aucun': 'nul'}

    # Apply mappings
    df['p'] = df['periode_construction'].map(p_map)
    df['rga'] = df['alea_argile'].map(level_map)
    df['type'] = df['type_inondation'].map(type_map)
    df['scen'] = df['scenario_inondation'].map(level_map)

    # 2. Total houses
    final_df = df.groupby('code_commune_insee')['nb_maisons'].sum().reset_index(name='total_maisons')

    # 3. RGA Pivot: rga_[period]_[level]
    rga_pivot = df.pivot_table(
        index='code_commune_insee', columns=['p', 'rga'],
        values='nb_maisons', aggfunc='sum', fill_value=0
    )
    rga_pivot.columns = [f"rga_{p}_{l}" for p, l in rga_pivot.columns]
    
    # 4. TRI Pivot: tri_[type]_[scenario]
    tri_pivot = df.pivot_table(
        index='code_commune_insee', columns=['type', 'scen'],
        values='nb_maisons', aggfunc='sum', fill_value=0
    )
    tri_pivot.columns = [f"tri_{t}_{s}" for t, s in tri_pivot.columns]

    # 5. Final Merge
    final_df = final_df.merge(rga_pivot, on='code_commune_insee', how='left')
    final_df = final_df.merge(tri_pivot, on='code_commune_insee', how='left')
    
    return final_df

# Run
df_flat = flatten_commune_data(OUTPUT_FILE_AGG)
df_flat.to_parquet('data/csv/france_communes_flat.parquet')


In [ ]:
df_flat.to_parquet('../csv/risques_communes_flat.parquet')

In [ ]:
df_flat.columns

In [ ]:
df_flat[df_flat.code_commune_insee == '33063']

# Source 4: GASPAR & CATNAT


## Base GASPAR


In [ ]:
# Le fichier source catanat_gaspar.csv a été transformé en gaspar_catnat.parquet pour synchro avec github (<25Mo)
catnat = pd.read_parquet('../csv/gaspar_catnat.parquet')
catnat.head()

In [ ]:
# "mouvements de terrain différentiels consécutifs à la sécheresse et à la réhydratation des sols" --> Sécheresse
catnat.lib_risque_jo.unique()

In [ ]:
catnat[catnat.lib_risque_jo == 'Sécheresse'].head(10)

## Recherche corrélation CATANAT & Batiments


Hypothèse: plus une commune a des maisons exposées à un risque fort (RGA élevé, % maisons construites > 1945) plus il y a des CATNAT de type Sécheresse enregistrés par la commune


In [ ]:
sec = catnat[catnat.num_risque_jo == 'SEC']

In [ ]:
sec_agg = sec.groupby('cod_commune').size().to_frame('count_sec').reset_index(names='code_commune_insee')
sec_agg.head()

In [ ]:
sec_agg.info()

In [ ]:
df_flat = pd.get_dummies(data=df_agg, columns=['alea_argile'], prefix='rga')
rga_cols = df_flat.columns[df_flat.columns.str.startswith('rga_')]
df_flat[rga_cols] = df_flat[rga_cols].multiply(df_flat['nb_maisons'], axis=0)
df_flat['constr_avant_1945'] = df_flat['periode_construction'].isin(['Avant 1915', '1916-1945', 'Inconnue'])
df_flat = df_flat.groupby(by=['code_commune_insee','constr_avant_1945']).sum().reset_index().drop(columns=['periode_construction'])
df_flat.head()

In [ ]:
# On doit penser à un score qui permet de montrer la prévalence du risque de RGA
#  Si année construction < 1945 + 0
#  Si après  alors si Nul +0, si Faible +1, si moyen +3, si For +5
#  Score =  Total / sum(nb_maisons)

df_flat.loc[df_flat['constr_avant_1945'], rga_cols] = 0
df_scored = df_flat.groupby('code_commune_insee').agg({'nb_maisons': 'sum', 'rga_Faible': 'sum', 'rga_Moyen': 'sum', 'rga_Fort': 'sum', 'rga_Nul': 'sum'}).reset_index()
df_scored['rga_score'] = (df_scored['rga_Faible'] + 3* df_scored['rga_Moyen'] + 5* df_scored['rga_Fort'] ) / (5* df_scored['nb_maisons'])
df_scored.head()

In [ ]:
df_corr = df_scored.merge(sec_agg, on="code_commune_insee")
df_corr.head()

In [ ]:
df_corr.describe()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

correlation = df_corr['rga_score'].corr(df_corr['count_sec'])
print(f"Pearson Correlation: {correlation:.4f}")

plt.figure(figsize=(10, 6))
sns.regplot(data=df_corr, x='rga_score', y='count_sec')
plt.title(f'Correlation between rga_score and count_sec (r={correlation:.2f})')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()